# Learning module 7 — Install the iOS driver and check prerequisites

**Small milestone:** add the iOS adapter to Appium and inspect its prerequisite report. This does not start the HTTP server or create a device session.

Installed and verified on **November 2, 2026 (America/Chicago)**:

| Setting | Observed value |
| --- | --- |
| Appium server package | `3.9.0` |
| XCUITest driver | `12.22.8` |
| Driver automation name | `XCUITest` |
| Driver location | Under `server/.tools/appium/node_modules/` |
| Doctor summary | `0 required fixes needed, 2 optional fixes possible` |

With the local Node runtime and Appium home configured, the installation command was:

```text
appium driver install xcuitest@12.22.8
```

Specifying the version prevents this command from selecting an unrelated newer driver major. The driver's Appium peer requirement is compatible with our server, and the published compatibility matrix includes this driver line for our iOS 27 target. [Driver installation](https://appium.github.io/appium-xcuitest-driver/latest/getting-started/installation/), [Compatibility matrix](https://appium.github.io/appium-xcuitest-driver/latest/getting-started/system-requirements/)

**The server package and driver are managed separately.** Our server npm manifest/lockfile were unchanged by the extension installation. The CLI-managed driver registration and its installation data live in the configured `APPIUM_HOME`. Restoring `server/package-lock.json` alone does not recreate that extension; the exact driver installation step is also required. [Extension management](https://appium.io/docs/en/latest/guides/managing-exts/)

Doctor found the Apple toolchain and iOS 27 Simulator SDK. Its two optional findings were missing `applesimutils` for certain Simulator permission operations and missing `ffmpeg` for video recording. Those capabilities are not requested by our current search-and-view scenario. Screenshot capture is a separate capability that we will verify during an actual session.

The real-device tunnel check reported that it did not run privileged operations, which is expected without root. Our selected target is a Simulator; this milestone did not create a real-device tunnel or apply fixes.

**Run this notebook's Node/Appium setup cells first.** The next cell verifies the existing driver registration and reruns Doctor. It does not install, update, or automatically fix tools. Doctor validates known prerequisites; its results do not prove that WebDriverAgent builds, an iOS session works, or Movies passes its assertions. [Doctor command](https://appium.io/docs/en/latest/reference/cli/extensions/#doctor)

## This notebook

Part **07 — Inspect the XCUITest registration and Doctor** of the learning sequence. [Notebook index](README.md).

Select the project's `.venv` Python kernel and run this notebook from top to bottom. Its Python setup is included locally; running another notebook in another kernel does not populate this one's variables or PATH. The supported working directories are the project root, `server/`, and `server/sample_scenarios/`.

**Execution scope:** Existing registration/prerequisite checks; no session.

Saved lesson outputs and embedded screenshots are historical snapshots preserved during the split on November 3, 2026. Newly included setup cells have no saved outputs. Rerunning an executable scenario creates fresh local artifacts; these continue living in `server/.tools/`, not beside the notebooks.

### Setup for this notebook

These source-only prerequisite cells are repeated intentionally so the lesson is runnable in a fresh kernel. They configure local values/environment and inspect tool versions; they do not launch an Appium server, create a session, or tap/type in the app. The original lessons remain linked for explanation.


In [ ]:
# Setup: Local Node activation (lesson 05)
import os
from pathlib import Path
import shutil
import subprocess

notebook_cwd = Path.cwd().resolve()
server_candidates = (notebook_cwd, notebook_cwd / "server", *notebook_cwd.parents)
server_dir = next((candidate for candidate in server_candidates
                   if (candidate / ".node-version").is_file()
                   and (candidate / "package.json").is_file()
                   and (candidate / "sample_scenarios").is_dir()), None)
if server_dir is None:
    raise RuntimeError("Run from the project root, server, or sample_scenarios folder.")

node_version = (server_dir / ".node-version").read_text().strip()
node_bin = server_dir / ".tools" / f"node-v{node_version}-darwin-arm64" / "bin"
if not (node_bin / "node").is_file():
    raise FileNotFoundError(f"Project-local Node is missing: {node_bin}")

path_entries = [entry for entry in os.environ.get("PATH", "").split(os.pathsep)
                if entry != str(node_bin)]
os.environ["PATH"] = os.pathsep.join([str(node_bin), *path_entries])

print("Selected Node:", shutil.which("node"))
print("Node version:", subprocess.check_output(["node", "--version"], text=True, timeout=10).strip())
print("npm version:", subprocess.check_output(["npm", "--version"], text=True, timeout=10).strip())


In [ ]:
# Setup: Local Appium activation (lesson 06)
import json
import os
import shutil
import subprocess

if "server_dir" not in globals() or "node_bin" not in globals():
    raise RuntimeError("Run the Node setup cell in this notebook first.")
if shutil.which("node") != str(node_bin / "node"):
    raise RuntimeError("Reactivate the project-local Node runtime first.")

appium_bin_dir = server_dir / "node_modules" / ".bin"
appium_executable = appium_bin_dir / "appium"
if not appium_executable.is_file():
    raise FileNotFoundError("The project-local Appium package is missing.")

os.environ["APPIUM_HOME"] = str(server_dir / ".tools" / "appium")
os.environ["npm_config_cache"] = str(server_dir / ".tools" / "npm-cache")
path_entries = [entry for entry in os.environ.get("PATH", "").split(os.pathsep)
                if entry != str(appium_bin_dir)]
os.environ["PATH"] = os.pathsep.join([str(appium_bin_dir), *path_entries])

expected_version = json.loads((server_dir / "package.json").read_text())["devDependencies"]["appium"]
actual_version = subprocess.check_output(
    [str(appium_executable), "--version"], cwd=server_dir, text=True, timeout=30
).strip()
if actual_version != expected_version:
    raise RuntimeError(f"Expected Appium {expected_version}, found {actual_version}.")

print("Selected Appium:", shutil.which("appium"))
print("Appium version:", actual_version)
print("Appium home:", os.environ["APPIUM_HOME"])


In [11]:
import json
import os
import subprocess

if "appium_executable" not in globals() or "server_dir" not in globals():
    raise RuntimeError("Run the Node/Appium setup cells in this notebook first.")
if os.environ.get("APPIUM_HOME") != str(server_dir / ".tools" / "appium"):
    raise RuntimeError("Rerun this notebook's Appium setup cell.")

registered = subprocess.run(
    [str(appium_executable), "driver", "list", "--installed", "--json"],
    cwd=server_dir, capture_output=True, text=True, timeout=30, check=True,
)
drivers = json.loads(registered.stdout)
xcuitest = drivers.get("xcuitest", {})
if xcuitest.get("version") != "12.22.8":
    raise RuntimeError("Expected the pinned XCUITest driver 12.22.8.")

print("Registered driver:", xcuitest["pkgName"])
print("Driver version:", xcuitest["version"])
print("Automation name:", xcuitest["automationName"])

doctor = subprocess.run(
    [str(appium_executable), "driver", "doctor", "xcuitest"],
    cwd=server_dir, capture_output=True, text=True, timeout=90,
)
print("Doctor exit code:", doctor.returncode)
print((doctor.stdout + doctor.stderr).strip())
if doctor.returncode != 0:
    raise RuntimeError("Doctor did not complete successfully; inspect its diagnostic output.")


Registered driver: appium-xcuitest-driver
Driver version: 12.22.8
Automation name: XCUITest
Doctor exit code: 0
dbug Appium Creating hash file directory: /Users/tarunkhurana/Engineering/projects/tarunkhurana2015/ai-certification-gen-academy/week3-project-automated-testing/server/.tools/appium/node_modules/.cache/appium
Running 8 doctor checks for the "xcuitest" driver
info Doctor ### Starting doctor diagnostics  ###
info Doctor  ✔ HOME is set to: /Users/tarunkhurana
info Doctor  ✔ Xcode is installed at '/Applications/Xcode.app/Contents/Developer'
info Doctor  ✔ Xcode Command Line Tools are installed and work properly
WARN Doctor  ✖ applesimutils are not installed
WARN Doctor  ✖ ffmpeg cannot be found
info Doctor  ✔ appium-ios-remotexpc is installed and can be imported. Remote XPC-based features are available for real devices (iOS/tvOS 18+).
info Doctor  ✔ The following Simulator SDKs are installed:
info Doctor 	→ Simulator - iOS 27.0
info Doctor 	→ Simulator - tvOS 27.0
info Doctor  ✔ 

### Read the driver and Doctor output, then pause

`automationName: XCUITest` is the driver selection we will later pass as `appium:automationName` when creating an iOS session. It is not a semantics identifier or a device ID.

| Diagnostic category | Result from this check | Interpretation |
| --- | --- | --- |
| Required prerequisites | Doctor reported zero required fixes | The prerequisites covered by these checks are satisfied. Session creation remains untested. |
| Optional permission tooling | `applesimutils` missing | Revisit if a scenario requires the associated Simulator permission operations. |
| Optional recording tooling | `ffmpeg` missing | Revisit if video recording is requested; do not confuse it with screenshot capture. |
| Real-device tunnel operation | Privileged steps were not run | This does not establish real-device readiness; the current target is a Simulator. |

The current driver writes the detailed diagnostics to stderr. Preserve and read its explicit summary, rather than treat an exit code or a generic closing message as proof that every optional capability is available. The initial `--json` invocation returned a numeric stdout value, so the learning cell deliberately displays the ordinary diagnostic text.

No optional tools were installed or fixes applied. Doctor is neither a business assertion check nor a security audit. The unresolved dependency findings recorded in module 6 remain open and need follow-up before server launch; this prerequisite report does not clear them.

**Pause here:** connect Appium package → registered XCUITest driver → known prerequisites → future device session. Our next small client-side tooling milestone is checking/installing the Python Appium client in the project's Python environment. Server startup, WebDriverAgent/session validation, and the actual Interstellar journey remain separate steps.

[Index](README.md) · [Previous lesson](06_appium_server_package.ipynb) · [Next lesson](08_python_client.ipynb)